# Extracción de datos del Mineduc


In [ ]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

### Importar datos

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager

# Configuración de Chrome
options = Options()

# Crear el navegador
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Abrir la página
driver.get("https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/")

# Maximizar la ventana
driver.maximize_window()

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 20)

departamento = wait.until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

print("Página cargada correctamente.")

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 20)

# Esperar a que aparezca el combo de departamentos
combo_departamento = wait.until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

# Seleccionar GUATEMALA
Select(combo_departamento).select_by_visible_text("GUATEMALA")

print("Departamento seleccionado.")

In [ ]:
import time

time.sleep(2)

# Seleccionar TODOS en municipio
combo_municipio = Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbMunicipio"
))

combo_municipio.select_by_visible_text("TODOS")

print("Municipio seleccionado.")

In [ ]:
Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbNivel"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbSector"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:ddlplan"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:ddlModalidad"
)).select_by_visible_text("TODOS")

print("Filtros seleccionados.")

In [ ]:
driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:IbtnConsultar"
).click()

print("Consulta enviada.")

## Leer tabla de centros educativos para Guatemala con Selenium

In [ ]:
tabla = driver.find_element(
    By.ID,
    "_ctl0_ContentPlaceHolder1_dgResultado"
)

print(tabla.get_attribute("outerHTML")[:500])

In [ ]:
from io import StringIO
import pandas as pd

html = tabla.get_attribute("outerHTML")

df = pd.read_html(StringIO(html))[0]

df.head()

## Limpieza de datos 

In [ ]:
# La primera fila contiene los encabezados
df.columns = df.iloc[0]

# Eliminar esa primera fila
df = df.iloc[1:].reset_index(drop=True)

df.head()

# Eliminar la primera columna (la del botón "Seleccionar Establecimiento")
df = df.drop(df.columns[0], axis=1)

df.head()


## Convertir datos a archivo CSV

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd
import time

def consultar(driver, departamento, nivel):

    wait = WebDriverWait(driver, 20)

    # Volver a la página principal
    driver.get("https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/")

    # Esperar que cargue el formulario
    wait.until(
        EC.presence_of_element_located(
            (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
        )
    )

    # Departamento
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )).select_by_visible_text(departamento)

    # Esperar que actualice el municipio
    time.sleep(2)

    # Municipio
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbMunicipio"
    )).select_by_visible_text("TODOS")

    # Nivel
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbNivel"
    )).select_by_visible_text(nivel)

    # Sector
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbSector"
    )).select_by_visible_text("TODOS")

    # Plan
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlplan"
    )).select_by_visible_text("TODOS")

    # Modalidad
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlModalidad"
    )).select_by_visible_text("TODOS")

    # Buscar
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:IbtnConsultar"
    ).click()

    # Esperar la tabla
    wait.until(
        EC.presence_of_element_located(
            (By.ID, "_ctl0_ContentPlaceHolder1_dgResultado")
        )
    )

    # Extraer HTML
    tabla = driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_dgResultado"
    )

    html = tabla.get_attribute("outerHTML")

    # Convertir a DataFrame
    df = pd.read_html(StringIO(html))[0]

    # Limpiar
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = df.drop(df.columns[0], axis=1)

    return df

In [ ]:
departamentos = [
    "GUATEMALA",
    "EL PROGRESO",
    "SACATEPEQUEZ",
    "CHIMALTENANGO",
    "ESCUINTLA",
    "SANTA ROSA",
    "SOLOLA",
    "TOTONICAPAN",
    "QUETZALTENANGO",
    "SUCHITEPEQUEZ",
    "RETALHULEU",
    "SAN MARCOS",
    "HUEHUETENANGO",
    "QUICHE",
    "BAJA VERAPAZ",
    "ALTA VERAPAZ",
    "PETEN",
    "IZABAL",
    "ZACAPA",
    "CHIQUIMULA",
    "JALAPA",
    "JUTIAPA"
]
niveles = [
    "DIVERSIFICADO"
]

todos = []

for depto in departamentos:

    for nivel in niveles:

        print(f"Consultando {depto} - {nivel}")

        df = consultar(driver, depto, nivel)

        todos.append(df)

In [ ]:
df_final = pd.concat(todos, ignore_index=True)

df_final.to_csv(
    "establecimientos.csv",
    index=False,
    encoding="utf-8-sig"
)